<a href="https://colab.research.google.com/github/fortunegbetox-tech/Code-ayant-le-score-maxi/blob/main/Code_route_ultra_ayant_le_score_maxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "albumentations", "-q"], check=True)

import gc, time, zipfile, warnings, sys
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from google.colab import drive, files

warnings.filterwarnings("ignore")
sys.setrecursionlimit(500)
torch.backends.cudnn.benchmark = True  # autotune cuDNN = +vitesse GPU

CFG = dict(
    seed         = 42,
    img_size     = 224,   # 224 → plus rapide que 256
    batch_size   = 128,   # batch plus grand = GPU mieux utilisé
    epochs       = 10,    # 10 epochs pour voir vite, augmente si besoin
    warmup_pct   = 0.10,
    lr           = 3e-4,
    lr_min       = 1e-6,
    weight_decay = 1e-4,
    label_smooth = 0.05,
    mixup_alpha  = 0.3,
    folds        = 3,     # 3 folds = 3× plus rapide que 5
    tta          = 4,     # 4 TTA suffisant, 8 = lent
    patience     = 4,
    num_workers  = 2,     # 2 workers = CPU charge pendant GPU calcule
    ckpt_dir     = "/content/ckpts",
    out_dir      = "/content/outputs",
)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
Path(CFG["ckpt_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["out_dir"]).mkdir(parents=True, exist_ok=True)

drive.mount("/content/drive", force_remount=True)
ROOT        = Path("/content/drive/MyDrive")
EXTRACT_DIR = Path("/content/img_fast")

if not EXTRACT_DIR.exists():
    print("Extraction…")
    with zipfile.ZipFile(ROOT / "Images.zip", "r") as z:
        z.extractall(EXTRACT_DIR)
    print("✓ Done")

IMG_INDEX = {f.stem: str(f) for f in EXTRACT_DIR.rglob("*.tif") if "__MACOSX" not in str(f)}
print(f"✓ {len(IMG_INDEX)} images | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

TRAIN_AUG = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Transpose(p=0.3),
    A.OneOf([
        A.RandomBrightnessContrast(0.2, 0.2, p=1.0),
        A.HueSaturationValue(5, 20, 20, p=1.0),
    ], p=0.4),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

VAL_AUG = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])


class RoadDataset(Dataset):
    def __init__(self, df, index, transform, is_test=False):
        self.df, self.index, self.transform, self.is_test = (
            df.reset_index(drop=True), index, transform, is_test)

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row  = self.df.iloc[i]
        path = self.index.get(str(row["Image_ID"]))
        img  = np.array(Image.open(path).convert("RGB")) if path \
               else np.zeros((CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)
        img  = self.transform(image=img)["image"]
        if self.is_test: return img, str(row["Image_ID"])
        return img, torch.tensor(float(row["Target"]), dtype=torch.float32)


def loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=True,
                      drop_last=shuffle, persistent_workers=True,
                      prefetch_factor=2)


def mixup(x, y, a=0.3):
    lam = np.random.beta(a, a)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x+(1-lam)*x[idx], lam*y+(1-lam)*y[idx]


class GeM(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1)*3.0)
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(1e-6).pow(self.p),(1,1)).pow(1/self.p)


class RoadModel(nn.Module):
    def __init__(self):
        super().__init__()
        bb = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.enc  = nn.Sequential(*list(bb.children())[:-2])
        self.pool = GeM()
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(2048), nn.Dropout(0.4),
            nn.Linear(2048, 512), nn.GELU(),
            nn.BatchNorm1d(512),  nn.Dropout(0.2),
            nn.Linear(512, 1),
        )
    def forward(self, x): return self.head(self.pool(self.enc(x)))


def bce(logits, y, eps=0.05):
    return F.binary_cross_entropy_with_logits(logits, y*(1-eps)+0.5*eps)


def get_sched(opt, total, warm):
    r = CFG["lr_min"] / CFG["lr"]
    def fn(s):
        if s < warm: return s / max(1, warm)
        p = (s-warm) / max(1, total-warm)
        return r + (1-r)*0.5*(1+np.cos(np.pi*p))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)


def train_epoch(model, dl, opt, scaler, sched, device):
    model.train()
    loss_sum = 0
    for x, y in dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        x, y = mixup(x, y, CFG["mixup_alpha"])
        opt.zero_grad(set_to_none=True)
        with autocast():
            loss = bce(model(x).view(-1), y, CFG["label_smooth"])
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        loss_sum += loss.item() * x.size(0)
    return loss_sum / len(dl.dataset)


@torch.no_grad()
def val_epoch(model, dl, device):
    model.eval()
    preds, labs = [], []
    for x, y in dl:
        with autocast():
            p = torch.sigmoid(model(x.to(device, non_blocking=True))).view(-1)
        preds.extend(p.float().cpu().tolist())
        labs.extend(y.tolist())
    return roc_auc_score(labs, preds) if len(set(labs))>1 else 0.5, np.array(preds)


TTA = [lambda x:x, lambda x:x.flip(-1), lambda x:x.flip(-2), lambda x:x.flip(-1).flip(-2)]

@torch.no_grad()
def tta_predict(model, dl, device):
    model.eval()
    out = []
    for x, _ in dl:
        x = x.to(device, non_blocking=True)
        with autocast():
            stack = torch.stack([torch.sigmoid(model(f(x))).view(-1) for f in TTA[:CFG["tta"]]])
        out.append(stack.mean(0).float().cpu().numpy())
    return np.concatenate(out)


def rank_avg(arrays):
    w = 1/len(arrays)
    out = np.zeros(len(arrays[0]))
    for a in arrays:
        out += w * pd.Series(a).rank(pct=True).values
    return out


def run_fold(fold, tr_idx, vl_idx, train_df, test_df, device):
    tr_dl   = loader(RoadDataset(train_df.iloc[tr_idx], IMG_INDEX, TRAIN_AUG), CFG["batch_size"], True)
    vl_dl   = loader(RoadDataset(train_df.iloc[vl_idx], IMG_INDEX, VAL_AUG),   CFG["batch_size"]*2, False)
    te_dl   = loader(RoadDataset(test_df, IMG_INDEX, VAL_AUG, is_test=True),   CFG["batch_size"]*2, False)

    model   = RoadModel().to(device)
    opt     = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    total   = len(tr_dl) * CFG["epochs"]
    sched   = get_sched(opt, total, int(total*CFG["warmup_pct"]))
    scaler  = GradScaler()

    best_auc, patience_cnt, ckpt = 0.0, 0, Path(CFG["ckpt_dir"])/f"f{fold}.pt"

    for ep in range(1, CFG["epochs"]+1):
        t0      = time.time()
        tr_loss = train_epoch(model, tr_dl, opt, scaler, sched, device)
        auc, _  = val_epoch(model, vl_dl, device)
        flag    = ""
        if auc > best_auc:
            best_auc, patience_cnt = auc, 0
            torch.save(model.state_dict(), ckpt)
            flag = " ✅"
        else:
            patience_cnt += 1
        print(f"  [{fold+1}] ep{ep:02d} | loss {tr_loss:.4f} | auc {auc:.4f} | {time.time()-t0:.0f}s{flag}")
        if patience_cnt >= CFG["patience"]:
            print("  ⏹ early stop"); break

    model.load_state_dict(torch.load(ckpt, map_location=device))
    _, vl_probs = val_epoch(model, vl_dl, device)
    te_probs    = tta_predict(model, te_dl, device)
    del model, opt, scaler; gc.collect(); torch.cuda.empty_cache()
    return best_auc, vl_probs, te_probs


def run():
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_df = pd.read_csv(ROOT/"Train.csv").dropna(subset=["Image_ID"]).reset_index(drop=True)
    test_df  = pd.read_csv(ROOT/"Test.csv").reset_index(drop=True)
    print(f"Train {len(train_df)} | Test {len(test_df)}\n")

    skf             = StratifiedKFold(CFG["folds"], shuffle=True, random_state=CFG["seed"])
    oof             = np.zeros(len(train_df))
    test_preds      = []
    fold_aucs       = []
    t0              = time.time()

    for fold, (tr, vl) in enumerate(skf.split(train_df, train_df["Target"])):
        print(f"{'─'*45}\n FOLD {fold+1}/{CFG['folds']}\n{'─'*45}")
        auc, vl_p, te_p = run_fold(fold, tr, vl, train_df, test_df, device)
        oof[vl] = vl_p[:len(vl)]
        test_preds.append(te_p)
        fold_aucs.append(auc)

    oof_auc = roc_auc_score(train_df["Target"], oof)
    print(f"\n{'═'*45}")
    print(f" OOF AUC : {oof_auc:.5f}  |  {(time.time()-t0)/60:.1f} min")
    for i,a in enumerate(fold_aucs): print(f" Fold {i+1} : {a:.5f}")
    print(f"{'═'*45}\n")

    oof_path = f"{CFG['out_dir']}/oof.csv"
    sub_path = f"{CFG['out_dir']}/submission.csv"

    pd.DataFrame({"Image_ID":train_df["Image_ID"],"Target":train_df["Target"],"oof":oof}).to_csv(oof_path, index=False)
    pd.DataFrame({"Image_ID":test_df["Image_ID"],"Target":rank_avg(test_preds)}).to_csv(sub_path, index=False)

    files.download(sub_path)
    files.download(oof_path)


run()

Mounted at /content/drive
✓ 10000 images | GPU: Tesla T4
Train 7000 | Test 3000

─────────────────────────────────────────────
 FOLD 1/3
─────────────────────────────────────────────
  [1] ep01 | loss 0.6067 | auc 0.9070 | 53s ✅
  [1] ep02 | loss 0.5127 | auc 0.9188 | 27s ✅
  [1] ep03 | loss 0.4928 | auc 0.8951 | 30s
  [1] ep04 | loss 0.4988 | auc 0.9370 | 30s ✅
  [1] ep05 | loss 0.4707 | auc 0.9328 | 29s
  [1] ep06 | loss 0.4193 | auc 0.9460 | 30s ✅
  [1] ep07 | loss 0.4393 | auc 0.9463 | 29s ✅
  [1] ep08 | loss 0.4214 | auc 0.9478 | 28s ✅
  [1] ep09 | loss 0.3992 | auc 0.9496 | 29s ✅
  [1] ep10 | loss 0.4249 | auc 0.9485 | 28s
─────────────────────────────────────────────
 FOLD 2/3
─────────────────────────────────────────────
  [2] ep01 | loss 0.6191 | auc 0.8934 | 30s ✅
  [2] ep02 | loss 0.4963 | auc 0.9030 | 29s ✅
  [2] ep03 | loss 0.4910 | auc 0.9295 | 29s ✅
  [2] ep04 | loss 0.4779 | auc 0.9255 | 29s
  [2] ep05 | loss 0.4584 | auc 0.9395 | 30s ✅
  [2] ep06 | loss 0.4367 | auc 0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>